# 03 Feature Engineering

This notebook creates business-ready variables: brand, revenue, discount metrics, date parts, and product categories.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

# File paths
orders_path = '/mnt/data/orders_cl.csv'
products_path = '/mnt/data/products_cl(1).csv'
orderlines_path = '/mnt/data/orderlines_qu.csv'
brands_path = '/mnt/data/brands(1).csv'

# Load datasets
orders_cl = pd.read_csv(orders_path)
products_cl = pd.read_csv(products_path)
orderlines_qu = pd.read_csv(orderlines_path)
brands_df = pd.read_csv(brands_path)

print('orders_cl:', orders_cl.shape)
print('products_cl:', products_cl.shape)
print('orderlines_qu:', orderlines_qu.shape)
print('brands_df:', brands_df.shape)

orders_clean = orders_cl.drop_duplicates().copy()
products_clean = products_cl.drop_duplicates(subset='sku').copy()
orderlines_clean = orderlines_qu.drop_duplicates().copy()
brands_clean = brands_df.drop_duplicates().copy()

orders_clean['created_date'] = pd.to_datetime(orders_clean['created_date'], errors='coerce')
orderlines_clean['date'] = pd.to_datetime(orderlines_clean['date'], errors='coerce')
orders_clean['total_paid'] = pd.to_numeric(orders_clean['total_paid'], errors='coerce')
products_clean['price'] = pd.to_numeric(products_clean['price'], errors='coerce')
orderlines_clean['unit_price'] = pd.to_numeric(orderlines_clean['unit_price'], errors='coerce')
orderlines_clean['product_quantity'] = pd.to_numeric(orderlines_clean['product_quantity'], errors='coerce')

orders_clean = orders_clean.dropna(subset=['order_id', 'created_date', 'total_paid', 'state'])
products_clean = products_clean.dropna(subset=['sku', 'name', 'price'])
orderlines_clean = orderlines_clean.dropna(subset=['id_order', 'sku', 'unit_price', 'product_quantity', 'date'])
products_clean = products_clean[products_clean['price'] > 0]
orderlines_clean = orderlines_clean[(orderlines_clean['unit_price'] > 0) & (orderlines_clean['product_quantity'] > 0)]


## 1. Create brand from SKU and enrich with brand name

In [ ]:
products_fe = products_clean.copy()
products_fe['brand_code'] = products_fe['sku'].str[:3]
products_fe = products_fe.merge(brands_clean, left_on='brand_code', right_on='short', how='left')
products_fe = products_fe.rename(columns={'long': 'brand_name'})

display(products_fe[['sku', 'brand_code', 'brand_name']].head())

## 2. Create rule-based product categories

In [ ]:
products_fe['text'] = (
    products_fe['name'].fillna('') + ' ' + products_fe['desc'].fillna('')
).str.lower()

def categorize_product(text):
    if ('iphone' in text) and ('case' not in text):
        return 'Smartphone'
    elif 'ipad' in text:
        return 'Tablet'
    elif ('macbook' in text) or ('imac' in text) or ('mac mini' in text) or ('mac pro' in text):
        return 'Computer'
    elif ('ssd' in text) or ('hard drive' in text) or ('external drive' in text) or ('nas' in text):
        return 'Storage'
    elif ('case' in text) or ('cable' in text) or ('adapter' in text) or ('charger' in text):
        return 'Peripheral'
    elif ('keyboard' in text) or ('mouse' in text) or ('headphone' in text) or ('speaker' in text):
        return 'Accessory'
    else:
        return 'Other'

products_fe['category'] = products_fe['text'].apply(categorize_product)
display(products_fe['category'].value_counts())

## 3. Build transaction-level analysis table

In [ ]:
sales_df = orderlines_clean.merge(
    products_fe[['sku', 'price', 'brand_name', 'category']],
    on='sku',
    how='left'
).merge(
    orders_clean[['order_id', 'created_date', 'state', 'total_paid']],
    left_on='id_order',
    right_on='order_id',
    how='left'
)

sales_df['revenue'] = sales_df['unit_price'] * sales_df['product_quantity']
sales_df['discount'] = sales_df['price'] - sales_df['unit_price']
sales_df['discount_pct'] = (sales_df['discount'] / sales_df['price']) * 100
sales_df['is_discounted'] = sales_df['unit_price'] < sales_df['price']

# Date parts
sales_df['order_year'] = sales_df['created_date'].dt.year
sales_df['order_month'] = sales_df['created_date'].dt.month
sales_df['order_month_name'] = sales_df['created_date'].dt.month_name()
sales_df['order_quarter'] = sales_df['created_date'].dt.quarter
sales_df['year_month'] = sales_df['created_date'].dt.to_period('M').astype(str)

display(sales_df.head())

## 4. Export engineered datasets

In [ ]:
feat_dir = '/mnt/data/eniac_project_notebooks/feature_data'
Path(feat_dir).mkdir(parents=True, exist_ok=True)

products_fe.to_csv(f'{feat_dir}/products_featured.csv', index=False)
sales_df.to_csv(f'{feat_dir}/sales_featured.csv', index=False)

print('Feature-engineered files exported to:', feat_dir)

## 5. Notes

Document any category rules you changed and why.